# Sweep по композиции перестановок $S_n$: $n=3,\ldots,10$

Этот Colab-ноутбук генерирует возобновляемые логи для проверки EDM-методов на нескольких $S_n$. Он использует токенизированное представление перестановок и предсказывает саму перестановку-произведение, поэтому не создаёт таблицу размера $|S_n|^2$. Результаты и checkpoints сохраняются в Google Drive.

## Перед запуском

В Colab выберите **Runtime → Change runtime type → T4 GPU**. Сначала выполните pilot для $S_3,\ldots,S_6$ и проверьте, что validation accuracy действительно проходит порог grokking. Только затем запускайте полный sweep.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys, torch
print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Выберите GPU runtime и перезапустите эту ячейку.'

REPO_URL = 'https://github.com/intsystems/2026-Project-202.git'
REPO_DIR = '/content/2026-Project-202'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
else:
    print('Repository already exists:', REPO_DIR)

WORKDIR = os.path.join(REPO_DIR, 'code', 'Grokking')
os.chdir(WORKDIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'einops', 'tqdm', 'pandas', 'scikit-learn', 'scipy'], check=True)
!git status --short

In [ ]:
# Загрузите актуальный sn_sweep_colab.py в панель Files Colab.
# Эта проверка не даст случайно запустить старую версию из git clone.
import os, shutil
UPLOADED_GENERATOR = '/content/sn_sweep_colab.py'
assert os.path.exists(UPLOADED_GENERATOR), 'Сначала загрузите новый sn_sweep_colab.py в /content'
shutil.copy2(UPLOADED_GENERATOR, os.path.join(WORKDIR, 'sn_sweep_colab.py'))

import importlib, sn_sweep_colab
importlib.reload(sn_sweep_colab)  # важно после обновления файла в Colab
from sn_sweep_colab import SweepConfig, run_sweep

DRIVE_ROOT = '/content/drive/MyDrive/grokking_sn_sweep'
# Один seed — отладка протокола. Для итоговой статьи используйте >=3 seeds.
PILOT = SweepConfig(
    output_root=DRIVE_ROOT,
    n_values=tuple(range(3, 11)),
    seeds=(42,),
    protocol_name='token_v4_genuine_gap10k',
    train_fraction=0.5,
    max_unique_pairs=14_400,
    max_epochs=3_000,
    min_steps=20_000,
    max_steps=200_000,
    log_every=50,
    checkpoint_every=1_000,
    num_workers=2,
)
run_sweep(PILOT)

Новый протокол записывается в подпапку `token_v4_genuine_gap10k`, поэтому старые несовместимые checkpoints не используются. Для всех $n$ архитектура и постоянный AdamW одинаковы; coordinate embeddings удалены, split равен 50/50. Успехом считается только реально измеренный разрыв не менее 10 000 шагов между устойчивой меморизацией и устойчивой генерализацией.

In [ ]:
# Если pilot с fraction=0.5 даёт раннюю генерализацию, калибруем только
# сложность данных. Архитектура и AdamW во всех попытках неизменны.
from sn_sweep_colab import calibrate_train_fraction, fractions_from_calibration
CALIBRATION = calibrate_train_fraction(PILOT, fractions=(0.5, 0.4, 0.3, 0.2))
display(CALIBRATION)
FRACTION_BY_N = fractions_from_calibration(CALIBRATION)
print('Accepted fractions:', FRACTION_BY_N)
# Для итогового эксперимента значения fraction следует зафиксировать заранее
# и проверить на новых seeds, не использованных в этой калибровке.

In [ ]:
# Быстрая проверка: должен существовать хотя бы один COMPLETED.json,
# а val_acc должен заметно изменяться; иначе сначала подберите протокол.
from pathlib import Path
import pandas as pd

for log_path in sorted(Path(DRIVE_ROOT).glob('token_v4_genuine_gap10k/S_*/seed_*/training_log.csv')):
    frame = pd.read_csv(log_path)
    print(log_path.parent, 'rows=', len(frame),
          'final val_acc=', round(frame.val_acc.iloc[-1], 4),
          'best val_acc=', round(frame.val_acc.max(), 4))

In [ ]:
# Основной запуск после успешного pilot.
# Перезапуск ячейки безопасен: checkpoint.pt продолжит незавершённый run,
# COMPLETED.json пропустит завершённый.
import importlib, sn_sweep_colab
importlib.reload(sn_sweep_colab)
from sn_sweep_colab import SweepConfig, run_sweep

FULL = SweepConfig(
    output_root=DRIVE_ROOT,
    n_values=tuple(range(3, 11)),
    seeds=(42, 43, 44),
    protocol_name='token_v4_genuine_gap10k',
    train_fraction=0.5,
    train_fraction_by_n=FRACTION_BY_N,
    max_unique_pairs=14_400,
    max_epochs=3_000,
    min_steps=20_000,
    max_steps=200_000,
    log_every=50,
    checkpoint_every=1_000,
    num_workers=2,
)
run_sweep(FULL)

## Локальный EDM-анализ после Colab

Синхронизируйте папку `grokking_sn_sweep` с Drive локально и запустите:

```powershell
python analyze_sn_sweep.py <путь-к-grokking_sn_sweep> results_sn_sweep --window-size 300 --stride 50
```

Итоги: `early_late_summary.csv`, траектории MLE по $n$ и сводные изменения FNN/Cao/Simplex/MLE.